# 08g THINGS-EEG1 without noise normalization

**Kaggle:** CPU, Internet on. **Inputs:** optionally the outputs of 08a, for the fidelity check.

The same conversion as 08a but without the whitening, stored in millivolts as float16. Test P-2 (08m) estimates each
new person's whitening from only their k labelled trials, which the whitened 08a arrays cannot provide. The output
goes to two datasets: the 22 development people and the 24 reserved people. Fidelity check: re-whitening two
development people must reproduce 08a.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '08g-unwhitened', 'cpu-convert'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
import json
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

try:
    import mne
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne"], check=True)
    import mne
from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset
from cross_subject_encoding.utils.preflight import preflight

ROLES = json.loads('{"sources": ["sub-04", "sub-07", "sub-11", "sub-12", "sub-13", "sub-14", "sub-15", "sub-17", "sub-20", "sub-21", "sub-24", "sub-25", "sub-26", "sub-29", "sub-30", "sub-34", "sub-35", "sub-38", "sub-39", "sub-46", "sub-48", "sub-49"], "targets": ["sub-02", "sub-03", "sub-05", "sub-08", "sub-09", "sub-10", "sub-16", "sub-19", "sub-22", "sub-27", "sub-28", "sub-31", "sub-32", "sub-33", "sub-36", "sub-37", "sub-40", "sub-41", "sub-42", "sub-43", "sub-44", "sub-45", "sub-47", "sub-50"]}')
OUT = {role: C.DATA_PROCESSED / f"eeg1_unwhitened_{role}" for role in ROLES}
WORK = Path("/kaggle/tmp/eeg1")
T0 = time.time()
minutes = lambda: f"{(time.time() - T0) / 60:.1f} min"
preflight("cross-subject-brain-encoding-08g-sources", inputs=[], need_gpu=False, expect_output_gb=8.0)
events = {}
for s in ROLES["sources"] + ROLES["targets"] + ["sub-02"]:
    p = WORK / s / f"{s}_task-rsvp_events.tsv"
    if not p.exists():
        e1.fetch(e1.S3 + f"{s}/eeg/{s}_task-rsvp_events.tsv", p)
    events[s] = e1.read_events(p)
train_keys = e1.training_order(events["sub-02"])  # the same shared order as 08a
print(len(train_keys), "training images |", minutes())

In [ ]:
rows = []
for role, people in ROLES.items():
    for s in people:
        t = time.time()
        files = e1.download_subject(s, WORK)
        data, ch, sf, onsets = e1.read_continuous(files["set"])
        person = e1.build_person(data, ch, sf, onsets, events[s], train_keys, whiten=False, scale=1e3)
        paths = e1.save_release(OUT[role], s, person)
        rows.append({"person": s, "role": role, **person["report"], "unit": "mV",
                     "max_abs_mV": float(np.abs(person["train"]).max()), "seconds": time.time() - t})
        del data, person
        shutil.rmtree(WORK / s, ignore_errors=True)
        pd.DataFrame(rows).to_csv(C.METRICS_DIR / "08g_audit.csv", index=False)
        print(s, role, rows[-1]["max_abs_mV"], f"{time.time() - t:.0f} s |", minutes())

In [ ]:
# Fidelity: re-whitening with all training trials must reproduce the 08a release (if attached)
check = []
for s in ROLES["sources"][:2]:
    old = e1.person_file(Path("/kaggle/input"), s, "training")
    if old is None:
        print("08a not attached: fidelity check skipped")
        break
    ref = np.load(old, allow_pickle=True).item()["preprocessed_eeg_data"].astype(np.float32)[:, 0]
    raw = np.load(e1.person_file(OUT["sources"], s, "training"), allow_pickle=True).item()["preprocessed_eeg_data"].astype(np.float32)[:, 0]
    w = e1.mvnn_whitener(raw)
    again = np.einsum("ij,njt->nit", w, raw)
    r = float(np.corrcoef(again.ravel(), ref.ravel())[0, 1])
    check.append({"person": s, "r_with_08a": r, "max_abs_diff": float(np.abs(again - ref).max())})
    print(check[-1])
status = {"status": "complete_unwhitened" if len(rows) == 46 else "incomplete", "people": len(rows),
          "fidelity": check, "fidelity_ok": all(c["r_with_08a"] > 0.999 for c in check) if check else None}
(C.METRICS_DIR / "08g_status.json").write_text(json.dumps(status, indent=2))
print(json.dumps(status, indent=1))

In [ ]:
for role in ROLES:
    export_to_dataset(slug=f"cross-subject-brain-encoding-08g-{role}", title=f"XSUB 08g unwhitened {role}",
                      files=sorted(OUT[role].glob("*.npy")) + [C.METRICS_DIR / "08g_status.json", C.METRICS_DIR / "08g_audit.csv"],
                      include_dirs=[], message=status["status"])